In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"

TABLE_NAMES = ["stores", "products", "customers", "orders", "order_items", "transactions"]
AUDIT_COLUMNS = ["_source_file", "_loaded_at"]

conn = sqlite3.connect(DB_PATH)
raw = {name: pd.read_sql(f"SELECT * FROM raw_{name}", conn).drop(columns=AUDIT_COLUMNS)
       for name in TABLE_NAMES}

cleaning_log = []

def log_step(table, action, rows_affected):
    cleaning_log.append({"table": table, "action": action, "rows_affected": int(rows_affected)})
    print(f"[{table}] {action}: {rows_affected:,} rows")

print("Loaded:", {name: len(df) for name, df in raw.items()})

Loaded: {'stores': 12, 'products': 300, 'customers': 4015, 'orders': 82089, 'order_items': 204230, 'transactions': 84390}


In [2]:
CATEGORY_MAP = {
    "electronics": "Electronics",
    "home & kitchen": "Home & Kitchen",
    "home&kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen",
    "fashion": "Fashion",
    "apparel": "Fashion",
    "grocery": "Grocery",
    "groceries": "Grocery",
    "sports": "Sports",
    "sport": "Sports",
}

def clean_products(df):
    df = df.copy()
    original = df["category"]

    df["category"] = df["category"].str.strip().str.lower().map(CATEGORY_MAP)

    unmapped = df["category"].isna()
    if unmapped.any():
        raise ValueError(f"Unknown categories: {sorted(original[unmapped].unique())}")
    log_step("products", "standardised category label", (df["category"] != original).sum())

    df["list_price"] = df["list_price"].astype(float)
    df["unit_cost"] = df["unit_cost"].astype(float)
    return df

products = clean_products(raw["products"])
products["category"].value_counts()

[products] standardised category label: 36 rows


category
Grocery           81
Fashion           78
Home & Kitchen    59
Electronics       53
Sports            29
Name: count, dtype: int64

In [3]:
def clean_customers(df):
    df = df.copy()

    before = len(df)
    df = df.drop_duplicates()
    log_step("customers", "dropped exact duplicate rows", before - len(df))

    blank_city = df["city"].str.strip() == ""
    df.loc[blank_city, "city"] = "Unknown"
    log_step("customers", "filled blank city with 'Unknown'", blank_city.sum())

    df["signup_date"] = pd.to_datetime(df["signup_date"], format="%Y-%m-%d")
    df["birth_year"] = df["birth_year"].astype(int)

    if df["customer_id"].duplicated().any():
        raise ValueError("customer_id is still not unique")
    return df

customers = clean_customers(raw["customers"])
print(customers.shape)
customers.dtypes

[customers] dropped exact duplicate rows: 15 rows
[customers] filled blank city with 'Unknown': 50 rows
(4000, 5)


customer_id               str
signup_date    datetime64[us]
city                      str
segment                   str
birth_year              int64
dtype: object

In [4]:
TODAY = pd.Timestamp.today().normalize()

def clean_orders(df, transactions):
    df = df.copy()

    before = len(df)
    df = df.drop_duplicates()
    log_step("orders", "dropped exact duplicate rows", before - len(df))

    df["has_customer"] = df["customer_id"].str.strip() != ""
    df = df.sort_values("has_customer", ascending=False)
    before = len(df)
    df = df.drop_duplicates(subset="order_id", keep="first")
    log_step("orders", "resolved same-ID conflicts (kept copy with customer)", before - len(df))
    df = df.drop(columns="has_customer").sort_values("order_id")

    df["order_ts"] = pd.to_datetime(df["order_ts"].str.strip(), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    bad_date = df["order_ts"].isna() | (df["order_ts"] > TODAY)

    paid = transactions[transactions["status"] == "Success"].drop_duplicates("order_id")
    payment_time = pd.to_datetime(paid.set_index("order_id")["transaction_ts"], format="%Y-%m-%d %H:%M:%S")
    df.loc[bad_date, "order_ts"] = df.loc[bad_date, "order_id"].map(payment_time)
    log_step("orders", "recovered bad order date from payment time", bad_date.sum())

    blank_customer = df["customer_id"].str.strip() == ""
    df.loc[blank_customer, "customer_id"] = None
    log_step("orders", "marked blank customer_id as missing", blank_customer.sum())

    if df["order_id"].duplicated().any():
        raise ValueError("order_id is still not unique")
    if df["order_ts"].isna().any() or (df["order_ts"] > TODAY).any():
        raise ValueError("some order dates are still missing or in the future")
    return df

orders = clean_orders(raw["orders"], raw["transactions"])
print(orders.shape)
print("Date range:", orders["order_ts"].min(), "to", orders["order_ts"].max())
orders[orders["order_id"] == "O0014050"]

[orders] dropped exact duplicate rows: 326 rows
[orders] resolved same-ID conflicts (kept copy with customer): 1 rows
[orders] recovered bad order date from payment time: 25 rows
[orders] marked blank customer_id as missing: 245 rows
(81762, 6)
Date range: 2024-01-02 16:34:00 to 2025-12-31 22:32:00


,order_id,customer_id,store_id,order_ts,channel,status
81895,O0014050,C02866,S004,2024-09-25 12:10:00,Online,Completed


In [5]:
def clean_order_items(df, products):
    df = df.copy()

    df["quantity"] = df["quantity"].astype(int)
    impossible = df["quantity"] <= 0
    df = df[~impossible]
    log_step("order_items", "removed lines with quantity 0 or less", impossible.sum())

    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    missing_price = df["unit_price"].isna()
    list_price = df["product_id"].map(products.set_index("product_id")["list_price"])
    df["unit_price"] = df["unit_price"].fillna(list_price)
    log_step("order_items", "filled missing unit_price with list price", missing_price.sum())

    df["discount_pct"] = df["discount_pct"].astype(float)
    df["is_returned"] = df["is_returned"] == "True"

    df["line_revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)).round(2)

    if df["unit_price"].isna().any():
        raise ValueError("some unit prices are still missing")
    return df

order_items = clean_order_items(raw["order_items"], products)
print(order_items.shape)
order_items.head(3)

[order_items] removed lines with quantity 0 or less: 60 rows
[order_items] filled missing unit_price with list price: 40 rows
(204170, 8)


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,line_revenue
0,I00000001,O0000001,P0249,1,1640.0,0.0,False,1640.0
1,I00000002,O0000002,P0249,1,1640.0,15.0,False,1394.0
2,I00000003,O0000002,P0046,2,140.0,10.0,False,252.0


In [6]:
empty_orders = ~orders["order_id"].isin(order_items["order_id"])
print("Orders with no valid items left:", empty_orders.sum())
orders[empty_orders]

Orders with no valid items left: 8


,order_id,customer_id,store_id,order_ts,channel,status
1272,O0001273,C02950,S002,2024-03-23 14:49:00,Online,Completed
17149,O0017150,C02426,S009,2024-10-17 17:28:00,Online,Completed
21893,O0021894,C03415,S004,2024-11-14 14:21:00,In-store,Completed
26872,O0026873,C03416,S002,2024-12-14 17:07:00,In-store,Completed
56345,O0056346,C00357,S004,2025-07-19 15:30:00,In-store,Completed
64926,O0064927,C01219,S005,2025-09-21 15:39:00,In-store,Completed
77830,O0077831,C02932,S003,2025-12-03 12:35:00,In-store,Completed
79455,O0079456,C03793,S011,2025-12-14 16:35:00,In-store,Completed


In [7]:
orders = orders[~empty_orders]
log_step("orders", "removed orders left with no valid items", empty_orders.sum())
print("Orders now:", len(orders))

[orders] removed orders left with no valid items: 8 rows
Orders now: 81754


In [8]:
def clean_transactions(df):
    df = df.copy()
    df["amount"] = df["amount"].astype(float)
    df["transaction_ts"] = pd.to_datetime(df["transaction_ts"], format="%Y-%m-%d %H:%M:%S")
    return df

def clean_stores(df):
    df = df.copy()
    df["opened_date"] = pd.to_datetime(df["opened_date"], format="%Y-%m-%d")
    return df

transactions = clean_transactions(raw["transactions"])
stores = clean_stores(raw["stores"])
print(transactions.dtypes)
print(stores.dtypes)

transaction_id               str
order_id                     str
payment_method               str
amount                   float64
transaction_ts    datetime64[us]
status                       str
dtype: object
store_id                  str
store_name                str
city                      str
region                    str
store_type                str
opened_date    datetime64[us]
dtype: object


In [9]:
from datetime import datetime, timezone

clean = {
    "stores": stores,
    "products": products,
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "transactions": transactions,
}

for name, df in clean.items():
    df.to_sql(f"clean_{name}", conn, if_exists="replace", index=False)
    print(f"clean_{name:<14} {len(df):>8,} rows")

log_df = pd.DataFrame(cleaning_log)
log_df["cleaned_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
log_df.to_sql("cleaning_log", conn, if_exists="replace", index=False)

REPORTS_DIR = PROJECT_ROOT / "reports"
log_df.to_csv(REPORTS_DIR / "cleaning_log.csv", index=False)

log_df

clean_stores               12 rows
clean_products            300 rows
clean_customers         4,000 rows
clean_orders           81,754 rows
clean_order_items     204,170 rows
clean_transactions     84,390 rows


,table,action,rows_affected,cleaned_at
0,products,standardised category label,36,2026-10-26T11:10:26+00:00
1,customers,dropped exact duplicate rows,15,2026-10-26T11:10:26+00:00
2,customers,filled blank city with 'Unknown',50,2026-10-26T11:10:26+00:00
3,orders,dropped exact duplicate rows,326,2026-10-26T11:10:26+00:00
4,orders,resolved same-ID conflicts (kept copy with cus...,1,2026-10-26T11:10:26+00:00
5,orders,recovered bad order date from payment time,25,2026-10-26T11:10:26+00:00
6,orders,marked blank customer_id as missing,245,2026-10-26T11:10:26+00:00
7,order_items,removed lines with quantity 0 or less,60,2026-10-26T11:10:26+00:00
8,order_items,filled missing unit_price with list price,40,2026-10-26T11:10:26+00:00
9,orders,removed orders left with no valid items,8,2026-10-26T11:10:26+00:00


In [10]:
checks = {
    "orders: order_id unique":
        "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM clean_orders",
    "customers: customer_id unique":
        "SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM clean_customers",
    "orders: no missing or future dates":
        "SELECT COUNT(*) FROM clean_orders WHERE order_ts IS NULL OR order_ts > date('now')",
    "order_items: no quantity below 1":
        "SELECT COUNT(*) FROM clean_order_items WHERE quantity < 1",
    "order_items: no missing price":
        "SELECT COUNT(*) FROM clean_order_items WHERE unit_price IS NULL",
    "products: only 5 categories":
        "SELECT COUNT(DISTINCT category) - 5 FROM clean_products",
    "customers: no blank city":
        "SELECT COUNT(*) FROM clean_customers WHERE TRIM(city) = ''",
    "orders: every order has items":
        "SELECT COUNT(*) FROM clean_orders WHERE order_id NOT IN (SELECT order_id FROM clean_order_items)",
}

results = []
for check, query in checks.items():
    problems = conn.execute(query).fetchone()[0]
    results.append({"check": check, "problems": problems, "passed": problems == 0})

pd.DataFrame(results)

,check,problems,passed
0,orders: order_id unique,0,True
1,customers: customer_id unique,0,True
2,orders: no missing or future dates,0,True
3,order_items: no quantity below 1,0,True
4,order_items: no missing price,0,True
5,products: only 5 categories,0,True
6,customers: no blank city,0,True
7,orders: every order has items,0,True


In [11]:
pd.read_sql("""
SELECT p.category, ROUND(SUM(oi.line_revenue)) AS revenue
FROM clean_order_items AS oi
JOIN clean_products AS p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY revenue DESC
""", conn)

,category,revenue
0,Electronics,288560038.0
1,Home & Kitchen,193362831.0
2,Fashion,128950157.0
3,Sports,57330619.0
4,Grocery,15045527.0
